**PLOTTING PREDICTIONS DISTRIBUTION**



In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import plotly.io as pio
pio.renderers.default='notebook'
import plotly.express as px
import datetime
import ast
import subprocess

In [ ]:
COLOR_PALLET_L2 = {
            'Other human': '#2986cc', # BLUE
            'Electro-mechanical': '#cc0000', # RED
            'Voice': '#6aa84f', #  green 6aa84f
            'Motorised transport': '#ffa500', # orange
            'Geonature': '#8e7cc3', # PURPLE
            'Animal': '#9b5f00', # BROWN
            'Music': '#d172a4', # PINK
            'Background': '#000000', # BLACK
            'Other Sounds': '#c9d631', # yellow
            'Social/communal': '#d8cbf8', # Light purple
            'Human movement': '#40b674', # light green 40b674
        }

COLOR_PALLET_L3 = {
    
}

In [70]:
def extract_location(file_path):
    file_name = os.path.basename(file_path)
    parts = file_name.split("_")
   
    if len(parts) > 2:
        return parts[2]
    else:
        return parts[0]
    
def postprocessing_df(df_preds):
    print(len(df_preds.columns))
    if len(df_preds.columns) != 5:
        df_preds.drop(columns=['classes_custom', 'probabilities_custom', 'sum_probs_custom', 'sum_probs_original'],inplace=True)
    else:
        df_preds.drop(columns=['classes_custom'],inplace=True)
    
    df_preds.rename(columns={'classes_original':'classes'}, inplace=True)
    df_preds['classes'] = df_preds['classes'].apply(ast.literal_eval)
    
    return df_preds

def remove_unnamed_columns(df_preds):
    df_preds = df_preds.loc[:, ~df_preds.columns.str.contains('^Unnamed')]
    return df_preds

def insert_time(df_preds):
    df_preds['date'] = pd.to_datetime(df_preds['datetime'])

    df_preds['year'] = df_preds.apply(lambda x: x['date'].year, axis= 1)
    df_preds['month'] = df_preds.apply(lambda x: x['date'].month, axis= 1)
    df_preds['day'] = df_preds.apply(lambda x: x['date'].day, axis= 1)

    df_preds['day_name'] = df_preds.apply(lambda x: x['date'].day_name(), axis= 1)
    df_preds['weekday'] = df_preds.apply(lambda x: x['date'].weekday(), axis= 1)
    
    df_preds['hour'] = df_preds.apply(lambda x: x['date'].hour, axis= 1)
    df_preds['hour_min'] = df_preds.apply(lambda x: str(x['date'].hour) + '_' + str(x['date'].minute),axis=1)
    
    df_preds['time'] = df_preds.apply(lambda x: x['date'].time(),axis=1)
    
    return df_preds

def print_end_start_date(df):
    start_date = df.sort_values('date')['date'].iloc[0]
    end_date = df.sort_values('date')['date'].iloc[-1]
    printing = 'Inicio Monitoreo: {} \nFinal Monitoreo: {}'.format(start_date,end_date)
    return printing

def index_and_explode(df):
    df.set_index('time',inplace=True)
    df = df.explode('classes')
    df['display_name'] = df['classes']
    df['number'] = 1
    
    return df

def merge_dataframes(df, union):
    df = df.merge(union,how='left',on='display_name')
    return df

def remove_label(df, column, label):
    df_no_label = df[df[column] != label]
    return df_no_label, label

# sort by time column called datetime
def sort_df(df):
    df.sort_values(by=['datetime'], inplace=True)
    return df

def output_dir(path: str):
    # get the abs path and remove the last element
    path = os.path.abspath(path).split("\\")[:-2]
    path = "/".join(path)
    
    visualization_dir = path + "/Visualizations"
    os.makedirs(visualization_dir, exist_ok=True)
    
    return visualization_dir

# get the last git tag version
def list_git_tags():
    try:
        tags = tags = subprocess.check_output(["git", "tag"]).strip().decode()
        return tags.split('\n')
    except subprocess.CalledProcessError:
        return None
    
def select_tag(tags):
    for i, tag in enumerate(tags):
        print(f"{i}: {tag}")
    choice = int(input("Select the tag to use: "))
    tag_selected = tags[choice]
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
        
    return tag_selected

def get_stable_version():
    tags = list_git_tags()
    # get the latest stable version
    tag_selected = tags[-1]
    print(f"Latest stable version with period: {tag_selected}")
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
    
    print(f"Latest stable version with underscore: {tag_selected}")
    
    return tag_selected

stable_version = get_stable_version()

Latest stable version with period: v1.0
Latest stable version with underscore: v1_0


In [50]:
yammnet_class_map = r"\\192.168.205.117\AAC_Server\PUERTOS\TEST_PROTOTYPE\20231211_SANTUR\5-Resultados\P1-CONTENEDORES-Reduce\Visualizations\audioset_ontology_test.csv"
df_audioset = pd.read_csv(yammnet_class_map)
df_audioset = df_audioset.drop(columns=['Brown_Level_1'])
df_audioset

,mid,display_name,iso_taxonomy,Brown_Level_2,Brown_Level_3
0,/m/09x0r,Speech,Voice,Voice,speech
1,/m/0ytgt,"Child speech, kid speaking",Voice,Voice,speech
2,/m/01h8n0,Conversation,Voice,Voice,speech
3,/m/02qldy,"Narration, monologue",Voice,Voice,speech
4,/m/0261r1,Babbling,Voice,Voice,speech
...,...,...,...,...,...
516,/m/07p_0gm,Throbbing,Background,Other Sounds,noise
517,/m/01jwx6,Vibration,Background,Other Sounds,noise
518,/m/07c52,Television,Background,Social/communal,domestic
519,/m/06bz3,Radio,Background,Social/communal,domestic


In [69]:
# print unique values in columns
print(f"Unique classes for Brown_Level_2 [ {len(df_audioset['Brown_Level_2'].unique())} ]")
print(df_audioset['Brown_Level_2'].unique())

print(f"\nUnique classes for Brown_Level_3 [ {len(df_audioset['Brown_Level_3'].unique())} ]")
print(df_audioset['Brown_Level_3'].unique())

Unique classes for Brown_Level_2 [ 11 ]
['Voice' 'Other human' 'Human movement' 'Animal' 'Electro-mechanical'
 'Music' 'Geonature' 'Other Sounds' 'Motorised transport'
 'Social/communal' 'Background']

Unique classes for Brown_Level_3 [ 34 ]
['speech' 'laughter' 'Crying' 'Voice' 'Other human' 'singing' 'footsteps'
 'Animal' 'Domestic Animal' 'Steam' 'Farm Animal' 'wildlife' 'Music'
 'wind' 'thunder' 'water' 'Other Sounds' 'roadway traffic'
 'marine traffic' 'alarms' 'rail traffic' 'air traffic' 'non motorized'
 'Engine' 'Tool' 'domestic' 'alarm' 'Machine' 'blasting/gunshot'
 'fireworks' 'Silence' 'Background people' 'Background Env' 'noise']


In [51]:
prediction_csv = r"\\192.168.205.117\AAC_Server\PUERTOS\TEST_PROTOTYPE\20231211_SANTUR\3-Medidas\5-Resultados\AUDIOMOTH\URBAN_Model\Predictions\Urban_Model_AUDIOMOTH_v1_0_window_1.0s_test.csv"
df_prediction = pd.read_csv(prediction_csv, converters={'classes_custom': eval})
# remove column
df_prediction = df_prediction.drop(columns=['classes_custom'])
df_prediction

,file,datetime,classes_original,probabilities_original
0,20231211_153450.WAV,2023-12-11 15:34:50,"['Vehicle', 'Rail transport', 'Train']","[0.7637054, 0.42956, 0.41159457]"
1,20231211_153450.WAV,2023-12-11 15:34:51,"['Vehicle', 'Rail transport', 'Train']","[0.34616104, 0.17398019, 0.1715938]"
2,20231211_153450.WAV,2023-12-11 15:34:52,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.36263642, 0.20404567, 0.11580885]"
3,20231211_153450.WAV,2023-12-11 15:34:53,"['Vehicle', 'Bicycle', 'Aircraft']","[0.37071517, 0.12847383, 0.056504596]"
4,20231211_153450.WAV,2023-12-11 15:34:54,"['Vehicle', 'Ship', 'Aircraft']","[0.31615925, 0.12974216, 0.094931796]"
...,...,...,...,...
895,20231211_153450.WAV,2023-12-11 15:49:45,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.9029716, 0.18985157, 0.17426494]"
896,20231211_153450.WAV,2023-12-11 15:49:46,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.8366037, 0.22521576, 0.1470383]"
897,20231211_153450.WAV,2023-12-11 15:49:47,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.7754757, 0.21422377, 0.18403114]"
898,20231211_153450.WAV,2023-12-11 15:49:48,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.770058, 0.34419206, 0.26134366]"


In [52]:
spl_csv = r"\\192.168.205.117\AAC_Server\PUERTOS\TEST_PROTOTYPE\20231211_SANTUR\5-Resultados\P1-CONTENEDORES-Reduce\SPL\P1-CONTENEDORES-Reduce_spl.csv"
df_spl = pd.read_csv(spl_csv)
df_spl = df_spl[['filename', 'date', 'LA', 'LC', 'LZ']]
df_spl

,filename,date,LA,LC,LZ
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12
1,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94
2,20231211_153450.wav,2023-12-11 15:34:52,70.26,75.70,75.83
3,20231211_153450.wav,2023-12-11 15:34:53,70.17,76.00,76.12
4,20231211_153450.wav,2023-12-11 15:34:54,70.20,75.78,75.86
...,...,...,...,...,...
895,20231211_153450.wav,2023-12-11 15:49:45,85.27,89.43,89.47
896,20231211_153450.wav,2023-12-11 15:49:46,86.23,89.99,90.06
897,20231211_153450.wav,2023-12-11 15:49:47,83.40,87.09,87.19
898,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75


# Union dataframes

In [53]:
#join the spl and prediction dataframes by date
df_spl['date'] = pd.to_datetime(df_spl['date'])
df_prediction['datetime'] = pd.to_datetime(df_prediction['datetime'])

# Merge the dataframes on the datetime columns
df_merged = pd.merge(df_spl, df_prediction, left_on='date', right_on='datetime')

df_merged = df_merged.drop(columns=['datetime', 'file'])

df_merged

,filename,date,LA,LC,LZ,classes_original,probabilities_original
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,"['Vehicle', 'Rail transport', 'Train']","[0.7637054, 0.42956, 0.41159457]"
1,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94,"['Vehicle', 'Rail transport', 'Train']","[0.34616104, 0.17398019, 0.1715938]"
2,20231211_153450.wav,2023-12-11 15:34:52,70.26,75.70,75.83,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.36263642, 0.20404567, 0.11580885]"
3,20231211_153450.wav,2023-12-11 15:34:53,70.17,76.00,76.12,"['Vehicle', 'Bicycle', 'Aircraft']","[0.37071517, 0.12847383, 0.056504596]"
4,20231211_153450.wav,2023-12-11 15:34:54,70.20,75.78,75.86,"['Vehicle', 'Ship', 'Aircraft']","[0.31615925, 0.12974216, 0.094931796]"
...,...,...,...,...,...,...,...
895,20231211_153450.wav,2023-12-11 15:49:45,85.27,89.43,89.47,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.9029716, 0.18985157, 0.17426494]"
896,20231211_153450.wav,2023-12-11 15:49:46,86.23,89.99,90.06,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.8366037, 0.22521576, 0.1470383]"
897,20231211_153450.wav,2023-12-11 15:49:47,83.40,87.09,87.19,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.7754757, 0.21422377, 0.18403114]"
898,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75,"['Vehicle', 'Aircraft', 'Fixed-wing aircraft, ...","[0.770058, 0.34419206, 0.26134366]"


In [54]:
df_merged['classes_original'] = df_merged['classes_original'].apply(ast.literal_eval)
df_merged['probabilities_original'] = df_merged['probabilities_original'].apply(ast.literal_eval)

df_exploded_classes = df_merged.explode('classes_original')
df_exploded = df_merged.apply(lambda x: x.explode() if x.name in ['classes_original', 'probabilities_original'] else x)

df_exploded


,filename,date,LA,LC,LZ,classes_original,probabilities_original
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Vehicle,0.763705
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Rail transport,0.42956
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Train,0.411595
1,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94,Vehicle,0.346161
1,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94,Rail transport,0.17398
...,...,...,...,...,...,...,...
898,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75,Aircraft,0.344192
898,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75,"Fixed-wing aircraft, airplane",0.261344
899,20231211_153450.wav,2023-12-11 15:49:49,77.43,81.04,81.09,Vehicle,0.593053
899,20231211_153450.wav,2023-12-11 15:49:49,77.43,81.04,81.09,Aircraft,0.103162


In [55]:
df_exploded['display_name'] = df_exploded['classes_original']
# merge classes with ontology
df_all = df_exploded.merge(df_audioset, how='left', on='display_name')
df_all

,filename,date,LA,LC,LZ,classes_original,probabilities_original,display_name,mid,iso_taxonomy,Brown_Level_2,Brown_Level_3
0,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Vehicle,0.763705,Vehicle,/m/07yv9,Roadway traffic,Motorised transport,roadway traffic
1,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Rail transport,0.42956,Rail transport,/m/06d_3,Rail traffic,Motorised transport,rail traffic
2,20231211_153450.wav,2023-12-11 15:34:50,70.23,76.01,76.12,Train,0.411595,Train,/m/07jdr,Rail traffic,Motorised transport,rail traffic
3,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94,Vehicle,0.346161,Vehicle,/m/07yv9,Roadway traffic,Motorised transport,roadway traffic
4,20231211_153450.wav,2023-12-11 15:34:51,69.91,75.82,75.94,Rail transport,0.17398,Rail transport,/m/06d_3,Rail traffic,Motorised transport,rail traffic
...,...,...,...,...,...,...,...,...,...,...,...,...
2695,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75,Aircraft,0.344192,Aircraft,/m/0k5j,Air traffic,Motorised transport,air traffic
2696,20231211_153450.wav,2023-12-11 15:49:48,80.02,83.68,83.75,"Fixed-wing aircraft, airplane",0.261344,"Fixed-wing aircraft, airplane",/m/0cmf2,Air traffic,Motorised transport,air traffic
2697,20231211_153450.wav,2023-12-11 15:49:49,77.43,81.04,81.09,Vehicle,0.593053,Vehicle,/m/07yv9,Roadway traffic,Motorised transport,roadway traffic
2698,20231211_153450.wav,2023-12-11 15:49:49,77.43,81.04,81.09,Aircraft,0.103162,Aircraft,/m/0k5j,Air traffic,Motorised transport,air traffic


**PLOTTING GLOBAL DASH BOARD**

In [149]:
# display_name, Brown_Level_1, Brown_Level_2, Brown_Level_3
display_name = 'display_name'
iso_taxonomy = 'iso_taxonomy'

brown_1 = 'Brown_Level_1'
brown_2 = 'Brown_Level_2'
brown_3 = 'Brown_Level_3'

class_to_plot = iso_taxonomy
original_classes = 'classes_original'

In [150]:
grouped_df = df_all.groupby([class_to_plot, original_classes]).size().reset_index(name='number')

# Plotting
fig = px.treemap(grouped_df, 
                 path=[class_to_plot, original_classes], 
                 values='number',
                 color=class_to_plot,
                 # color_discrete_map=COLLOR_PALLET 
                 color_discrete_map=COLOR_PALLET_L2
                )

fig.update_layout(title='Title')
fig.show()

In [80]:
def leq(levels):
    """Get the Leq from a list of levels
    Args:
        levels: list of levels
        
    Returns:
        leq: Leq
    """
    levels = levels[~np.isnan(levels)]
    l = np.array(levels)
    return 10*np.log10(np.mean(np.power(10,l/10)))

In [135]:
grouped_df = df_all.groupby(['Brown_Level_3', 'classes_original']).agg(
    number=('classes_original', 'size'), 
    LA_Arit_Avg=('LA', 'mean')
).reset_index()

# Plotting
fig = px.treemap(grouped_df, 
                 path=['Brown_Level_3', 'classes_original'], 
                 values='number',
                 color='LA_Arit_Avg',
                 color_continuous_scale='RdYlGn',
                 # color_discrete_map=COLOR_PALLET_L2 
                )

fig.update_layout(title='Arithmetic Average Sound Pressure Level (LAeq)')
fig.show()

In [141]:
grouped_df = df_all.groupby([class_to_plot, original_classes]).agg(
    number=(original_classes, 'size'),
    LA_Power_Avg=('LA', lambda x: leq(x))
).reset_index()

fig = px.treemap(grouped_df, 
                 path=[class_to_plot, original_classes], 
                 values='number',
                 color='LA_Power_Avg',
                 color_continuous_scale='RdYlGn',
                )

fig.update_layout(title='Power Average Sound Pressure Level (LAeq)')
fig.show()

In [143]:
# Adjust the aggregation to focus only on 'Brown_Level_2'
grouped_df = df_all.groupby(class_to_plot).agg(
    number=(original_classes, 'size'),
    LA_Power_Avg=('LA', lambda x: leq(x))
).reset_index()

fig = px.treemap(grouped_df, 
                 path=[class_to_plot], 
                 values='number',
                 color='LA_Power_Avg',
                 color_continuous_scale='RdYlGn'
                )

fig.update_layout(title='Power Average Sound Pressure Level (LAeq)')
fig.show()

In [144]:
grouped_df = df_all.groupby([class_to_plot, original_classes]).agg(
    number=(original_classes, 'size'),
    LA_Power_Avg=('LA', lambda x: leq(x))
).reset_index()

# Calculate total counts for normalization
grouped_df['total'] = grouped_df.groupby(class_to_plot)['number'].transform('sum')

# Calculate percentage
grouped_df['percentage'] = (grouped_df['number'] / grouped_df['total'] * 100).round(2)

# Plotting
fig = px.treemap(grouped_df, 
                 path=[class_to_plot, original_classes], 
                 values='number',
                 color='LA_Power_Avg',
                 color_continuous_scale='RdYlGn',
                 hover_data=['number', 'LA_Power_Avg', 'percentage']
                )

fig.update_layout(title='Sound Class Distribution by LA Power Average with Percentage')
fig.data[0].texttemplate = "%{label}<br>%{customdata[2]}%"
fig.show()

In [148]:
grouped_df = df_all.groupby([class_to_plot, original_classes]).agg(
    number=(original_classes, 'size'),
    LA_Power_Avg=('LA', lambda x: leq(x))
).reset_index()

overall_total = grouped_df['number'].sum()

grouped_df['total_brown_level_2'] = grouped_df.groupby(class_to_plot)['number'].transform('sum')
grouped_df['percentage_brown_level_2'] = (grouped_df['total_brown_level_2'] / overall_total * 100).round(2)

grouped_df['percentage_display'] = grouped_df['percentage_brown_level_2'].astype(str) + '%'
grouped_df['percentage_display'] = grouped_df.groupby(class_to_plot)['percentage_display'].transform('first')

fig = px.treemap(grouped_df, 
                 path=[class_to_plot, original_classes], 
                 values='number',
                 color='LA_Power_Avg',
                 color_continuous_scale='RdYlGn',
                 hover_data=['number', 'LA_Power_Avg', 'percentage_brown_level_2']
                )
fig.update_layout(title='Sound Class Distribution by LA Power Average')
fig.data[0].texttemplate = "%{label}<br>%{customdata[2]}%"
fig.data[0].texttemplate = "%{label}<br>%{customdata[2]}%"
fig.show()